# Build Dataset Generation Config

This notebook processes official energy data from:

- EPE (Empresa de Pesquisa Energética)
- ANEEL (Agência Nacional de Energia Elétrica)

The goal is to generate the probabilistic configuration files used by the synthetic dataset generator.

Outputs:
- consumption_profiles.csv
- energy_source_distribution.csv
- company_size_distribution_by_usage.csv
- usage_distribution_by_state.csv
- seasonality_state_class_month.csv

Clona Projeto direto do github para facilitar

In [ ]:
# !git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

importa dados oficiais da empresa de pesquisa energetica e faz EDA + ETL para criar o dataset sintetico ainda !

Abaixo e o data set de consumo por categoria e estado o consumo esta em MW mas abaixo sera convertido para kwh

In [ ]:
import pandas as pd

epe = pd.read_csv('../data/raw/epe_industrial_consumption_by_state.csv',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
epe.head()

In [ ]:
epe['DataExcel'] = pd.to_datetime(epe['DataExcel'], dayfirst=True)

In [ ]:
epe.info()

In [ ]:
epe['SetorIndustrial'].unique()

In [ ]:
df = epe.copy()

In [ ]:
df['year'] = df['DataExcel'].dt.year
df['month'] = df['DataExcel'].dt.month

In [ ]:
df_2025 = df[df['year'] == 2025].copy()
df_2025.drop(columns=['DataVersao', 'DataExcel'], inplace=True)

In [ ]:
df_2025.head(5)

In [ ]:
df_grouped = df.groupby('SetorIndustrial')['Consumo'].mean().reset_index()

In [ ]:
df_grouped = df_grouped.rename(columns={'Consumo':'Consumo_MWh'})

In [ ]:
df_grouped.head()

In [ ]:
df_grouped['Consumo_kWh'] = (df_grouped['Consumo_MWh']) * 1000

In [ ]:
df_grouped.head()

In [ ]:
df_grouped.to_csv('media_consumo_indutria_2025.csv')

Abertura do dados oficial consumo mensal por categoria a escala de energia esta em MW mas sera convertido em kwh.

Tambem foi feito o EDA e ETL para criação do dataset sintetico !

In [ ]:
import pandas as pd

epe_categoria = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_2025 = df[df['year'] == 2025].copy()
df_2025.drop(columns=['DataVersao', 'DataExcel'], inplace=True)

In [ ]:
epe_categoria.info()

In [ ]:
epe_categoria['DataExcel'] = pd.to_datetime(epe_categoria['DataExcel'], dayfirst=True)

In [ ]:
df = epe_categoria.copy()

In [ ]:
df['year'] = df['DataExcel'].dt.year
df['month'] = df['DataExcel'].dt.month

In [ ]:
df_2025 = df[df['year'] == 2025]

In [ ]:
df_2025['Consumo'] = (
    df_2025['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_2025['Consumidores'] = (
    df_2025['Consumidores']
    .astype(str)
    .str.replace('.', '', regex=False)
    .astype(float)
)

In [ ]:
df_2025.info()

In [ ]:
df_2025.head()

In [ ]:
df_2025.drop(columns=['Data', 'DataVersao'], inplace=True)

In [ ]:
df_2025 = df_2025.rename(columns={'Consumo':'Consumo_MWh'})

In [ ]:
df_simplificado = (
    df_2025
    .groupby('Classe')
    .agg({
        'Consumo_MWh': 'sum',
        'Consumidores': 'sum'
    })
)

df_simplificado['consumo_medio_MWh'] = df_simplificado['Consumo_MWh'] / df_simplificado['Consumidores']

df_simplificado = df_simplificado.sort_values('Consumo_MWh', ascending=False)

In [ ]:
df_simplificado['Consumo_kWh'] = (df_simplificado['Consumo_MWh']) * 1000
df_simplificado['consumo_medio_KWh'] = (df_simplificado['consumo_medio_MWh']) * 1000

In [ ]:
df_simplificado = df_simplificado[[
    'Consumidores',
    'Consumo_MWh',
    'Consumo_kWh',
    'consumo_medio_MWh',
    'consumo_medio_KWh'
]]

In [ ]:
df_simplificado

In [ ]:
df_simplificado.to_csv('consumo_medio_categoria_2025.csv')

Verificando a sanidade dos dados

In [ ]:
df_grouped.head()

In [ ]:
df_grouped.info()

In [ ]:
df_simplificado.head()

In [ ]:
df_simplificado.info()

Criação do consumption_profile.csv

In [ ]:
df_profiles = df_simplificado.copy()

In [ ]:
df_profiles.head()

In [ ]:
def get_sigma(classe):
    if classe == 'Industrial':
        return 0.6
    elif classe == 'Comercial':
        return 0.4
    elif classe == 'Residencial':
        return 0.3
    else:
        return 0.35

df_profiles['sigma'] = df_profiles.index.map(get_sigma)

In [ ]:
map_usage = {
    'Industrial': 'industrial',
    'Comercial': 'commercial',
    'Residencial': 'residential',
    'Rural': 'agriculture',
    'Outros': 'other'
}

df_profiles['usage_type'] = df_profiles.index.map(map_usage)

In [ ]:
import numpy as np

df_profiles['mu'] = np.log(df_profiles['consumo_medio_KWh'])

In [ ]:
df_final = df_profiles.copy()
df_final['fuel_type'] = 'electric'
df_final['distribution_type'] = 'lognormal'
df_final['param_1'] = df_profiles['mu']
df_final['param_2'] = df_profiles['sigma']
df_final['param_1_name'] = 'mu'
df_final['param_2_name'] = 'sigma'
df_final['unit'] = 'kWh'
df_final['is_energy_based'] = True

In [ ]:
df_final.head()

In [ ]:
df_export = df_final[[
    'usage_type',
    'fuel_type',
    'distribution_type',
    'param_1',
    'param_2',
    'param_1_name',
    'param_2_name',
    'unit',
    'is_energy_based'
]].copy()

In [ ]:
df_export.head()

In [ ]:
df_export.to_csv('v2_consumption_profiles.csv', index=False)

transformar config em dados sinteticos reais

carergando profile

In [ ]:
import pandas as pd
import numpy as np

profiles = pd.read_csv('/content/v2_consumption_profiles.csv')

gerar função

In [ ]:
def generate_consumption(profile_row):

    if profile_row['distribution_type'] == 'lognormal':
        mu = profile_row['param_1']
        sigma = profile_row['param_2']

        value = np.random.lognormal(mu, sigma)

        # LIMITADOR
        max_value = 10 * np.exp(mu)
        value = min(value, max_value)

        return value

    else:
        raise ValueError("Distribuição não suportada")

gerar evento

In [ ]:
usage_probs = {
    'residential': 0.6,
    'commercial': 0.2,
    'industrial': 0.1,
    'agriculture': 0.05,
    'other': 0.05
}

In [ ]:
def generate_event(profiles):

    weights = profiles['usage_type'].map(usage_probs)

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    return {
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption
    }

gerar dataset

In [ ]:
def generate_dataset(n, profiles):
    data = []

    for _ in range(n):
        event = generate_event(profiles)
        data.append(event)

    return pd.DataFrame(data)

test

In [ ]:
df = generate_dataset(10000, profiles)

In [ ]:
df.head()

In [ ]:
df.to_csv('df_gerado_teste.csv')

In [ ]:
df['usage_type'].value_counts(normalize=True)

In [ ]:
df.groupby('usage_type')['energy_kwh'].describe()

In [ ]:
df['energy_kwh'].max()

In [ ]:
(df['energy_kwh'] < 0).sum()

In [ ]:
import matplotlib.pyplot as plt

df[df['usage_type']=='industrial']['energy_kwh'].hist(bins=50)
plt.title('Industrial')
plt.show()

criando o energy _source_distribution.csv

In [ ]:
df_aneel = pd.read_csv('../data/raw/aneel_generation.csv',
                       encoding='latin-1',
                       sep=';',
                       decimal=',')

In [ ]:
df_aneel.info()

In [ ]:
df_aneel.head()

In [ ]:
df_aneel['AnoReferencia'].unique()

In [ ]:
df_aneel = df_aneel[df_aneel['AnoReferencia']==2025]

In [ ]:
df_aneel['SigTipoGeracao'].unique()

In [ ]:
df_grouped = df_aneel.groupby('SigTipoGeracao')['MdaPotenciaInstaladaKW'].sum()

In [ ]:
df_grouped.head()

converter para probabilidade

In [ ]:
df_dist = df_grouped / df_grouped.sum()

transformar e dataframe

In [ ]:
df_dist.info()

In [ ]:
df_grouped = df_aneel.groupby('SigTipoGeracao')['MdaPotenciaInstaladaKW'].sum().reset_index()

df_grouped.columns = ['energy_source', 'total_generation']

In [ ]:
df_grouped['probability'] = df_grouped['total_generation'] / df_grouped['total_generation'].sum()

df_dist = df_grouped[['energy_source', 'probability']]
df_dist = df_dist.copy()

In [ ]:
df_dist.head()

In [ ]:
df_dist['probability'].sum()

In [ ]:
map_sources = {
    'UHE': 'hydro',      # Hidrelétrica grande
    'PCH': 'hydro',      # Pequena central hidrelétrica
    'CGH': 'hydro',      # Central geradora hidráulica

    'EOL': 'wind',       # Eólica

    'UFV': 'solar',      # Solar fotovoltaica

    # se aparecer depois:
    'UTE': 'thermal',    # termoelétrica (genérico)
    'UTN': 'nuclear',    # nuclear
}
df_dist['energy_source'] = df_dist['energy_source'].map(map_sources)

In [ ]:
df_dist = df_dist.groupby('energy_source')['probability'].sum().reset_index()

In [ ]:
df_dist['probability'] = df_dist['probability'] / df_dist['probability'].sum()

In [ ]:
df_dist = df_dist.sort_values('probability', ascending=False)

In [ ]:
df_dist

In [ ]:
df_dist.to_csv('energy _source_distribution.csv')

Inserir a emissão co2

In [ ]:
emission_factors = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

In [ ]:
def calculate_emission(energy_kwh, energy_source, emission_df):

    row = emission_df.loc[
        emission_df['energy_source'] == energy_source
    ]

    if row.empty:
        raise ValueError(f"Fonte não encontrada: {energy_source}")

    factor = row['emission_factor'].values[0]

    return energy_kwh * factor

In [ ]:
def sample_energy_source(dist_df):
    return dist_df.sample(1, weights=dist_df['probability']).iloc[0]['energy_source']

In [ ]:
def normalize_usage_types(series):

    mapping = {
        'residencial': 'residential',
        'rural': 'agriculture',
        'outros': 'other',
        'industrial': 'industrial',
        'commercial': 'commercial'
    }

    return (
        series
        .str.strip()
        .str.lower()
        .replace(mapping)
    )

In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    usage_types = normalize_usage_types(profiles['usage_type'])

    weights = usage_types.map(usage_probs)

    if weights.sum() == 0:
        raise ValueError("usage_probs não corresponde aos usage_type do profile")

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    energy_source = sample_energy_source(energy_dist)

    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
import pandas as pd
profiles = pd.read_csv('../data/processed/v2_consumption_profiles.csv')

energy_dist = pd.read_csv('../data/processed/v2_energy _source_distribution.csv')

emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

In [ ]:
event = generate_event(profiles, energy_dist, emission_df)
event

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df.head()

Adicionando ID e datas

In [ ]:
company_id = f"C{np.random.randint(1000,9999)}"

In [ ]:
date = pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365), unit='D')

In [ ]:
states = [
'SP','MG','RJ','BA','PR','RS','SC','GO','PE','CE',
'PA','MT','ES','DF','MS','MA','RN','PB','AL','PI',
'RO','SE','TO','AC','AP','RR','AM'
]

state_weights = [
0.22,0.10,0.09,0.07,0.06,0.06,0.04,0.04,0.04,0.04,
0.03,0.03,0.02,0.02,0.02,0.02,0.015,0.015,0.01,0.01,
0.008,0.008,0.006,0.005,0.005,0.003,0.02
]


In [ ]:
len(states), len(state_weights)

In [ ]:
state_weights = np.array(state_weights)
state_weights = state_weights / state_weights.sum()
state_weights

In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    usage_types = normalize_usage_types(profiles['usage_type'])

    weights = usage_types.map(usage_probs)

    if weights.sum() == 0:
        raise ValueError("usage_probs não corresponde aos usage_type do profile")

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    energy_source = sample_energy_source(energy_dist)

    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'company_id': f"C{np.random.randint(100000,999999)}",
        'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0,365), unit='D'),
        'state': np.random.choice(states, p=state_weights),
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df

Adicionando regras mais realistas, do jeito que esta esta pode criar industria pesadas na msm probabilidade entre ACRE e São Paulo

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe.columns

In [ ]:
df_epe.head(5)

In [ ]:
df_epe.info()

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_sector_state = (
    df_epe
    .groupby(['UF','Classe'])['Consumo']
    .sum()
    .reset_index()
)

In [ ]:
df_sector_state.head(20)

In [ ]:
df_sector_state['probability'] = (
    df_sector_state
    .groupby('UF')['Consumo']
    .transform(lambda x: x / x.sum())
)

In [ ]:
usage_distribution_by_state = df_sector_state.pivot(
    index='UF',
    columns='Classe',
    values='probability'
)

In [ ]:
usage_distribution_by_state.to_csv('usage_distribution_by_state.csv')

distribuição por estado

In [ ]:
df_epe

In [ ]:
state_consumption = df_epe.groupby("UF")["Consumo"].sum()

state_distribution = state_consumption / state_consumption.sum()


In [ ]:
state_distribution

In [ ]:
state_distribution.to_csv("v2_state_distribution.csv")

In [ ]:
df_state = pd.read_csv("v2_state_distribution.csv")

states = df_state["UF"]
state_weights = df_state["Consumo"]


In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    # sorteia estado
    state = np.random.choice(states, p=state_weights)

    # pega probabilidades de setor para esse estado
    usage_probs_state = usage_distribution_by_state.loc[state].values
    usage_probs_state = usage_probs_state / usage_probs_state.sum()

    # sorteia setor
    usage_types = profiles['usage_type'].unique()

    usage_type = np.random.choice(
        usage_types,
        p=usage_probs_state
    )

    # filtra profiles para esse setor
    profiles_subset = profiles[profiles['usage_type'] == usage_type]

    # escolhe linha de profile
    row = profiles_subset.sample(1).iloc[0]

    # gera consumo
    consumption = generate_consumption(row)

    # sorteia fonte de energia
    energy_source = sample_energy_source(energy_dist)

    # calcula CO2
    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'company_id': f"C{np.random.randint(100000,999999)}",
        'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0,365), unit='D'),
        'state': np.random.choice(states, p=state_weights),
        'usage_type': usage_type,
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)

In [ ]:
df.info()

Extraindo a probabilidade do tramanho do negocio small, medium e large. Para criar dataset mais realista !

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_epe['DataExcel'] = pd.to_datetime(df_epe['DataExcel'], dayfirst=True)

In [ ]:
df_epe['year'] = df_epe['DataExcel'].dt.year
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
df_2025 = df_epe[df_epe['year'] == 2025].copy()

In [ ]:
df_2025

In [ ]:
df_2025['DataExcel'].unique()

In [ ]:
df_2025['year'].unique()

In [ ]:
df_epe = df_2025.copy()

In [ ]:
def classify_company_size(row):

    usage = row['Classe']
    energy = row['Consumo']

    if usage == 'Industrial':
        if energy < 20000:
            return 'small'
        elif energy < 100000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Comercial':
        if energy < 5000:
            return 'small'
        elif energy < 30000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Residencial':
        return 'small'

    if usage == 'Rural':
        if energy < 10000:
            return 'small'
        elif energy < 50000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Outros':
        if energy < 10000:
            return 'small'
        elif energy < 50000:
            return 'medium'
        else:
            return 'large'

In [ ]:
df_epe['company_size'] = df_epe.apply(classify_company_size, axis=1)

In [ ]:
df_epe['company_size'].value_counts()

In [ ]:
size_dist = (
    df_epe
    .groupby('Classe')['company_size']
    .value_counts(normalize=True)
    .reset_index(name='probability')
)

In [ ]:
size_dist

In [ ]:
size_dist.to_csv(
    'v2_company_size_distribution_by_usage.csv',
    index=False
)

calculando a variancia por mes, meses podem impactar no consumo de energia !

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_epe['DataExcel'] = pd.to_datetime(df_epe['DataExcel'], dayfirst=True)

In [ ]:
df_epe['year'] = df_epe['DataExcel'].dt.year
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
df_2025 = df_epe[df_epe['year'] == 2025].copy()

In [ ]:
df_2025

In [ ]:
df_epe = df_2025.copy()

In [ ]:
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
monthly_stats = (
    df_epe
    .groupby(['UF','Classe','month'])['Consumo']
    .agg(['mean','var'])
    .reset_index()
)

In [ ]:
annual_mean = (
    df_epe
    .groupby(['UF','Classe'])['Consumo']
    .mean()
    .reset_index()
    .rename(columns={'Consumo':'annual_mean'})
)

In [ ]:
monthly_stats = monthly_stats.merge(
    annual_mean,
    on=['UF','Classe']
)

In [ ]:
monthly_stats['seasonal_factor'] = (
    monthly_stats['mean'] /
    monthly_stats['annual_mean']
)

In [ ]:
monthly_stats

In [ ]:
monthly_stats = monthly_stats[[
    'UF',
    'Classe',
    'month',
    'seasonal_factor',
    'var'
]]

monthly_stats.to_csv(
    'v2_seasonality_state_class_month.csv',
    index=False
)

Gerando df completo

In [ ]:
!ls

In [ ]:
from pathlib import Path

ROOT = Path().resolve().parent

In [ ]:
!pwd

In [ ]:
!git pull

import csvs

In [ ]:
import pandas as pd
import numpy as np

profiles = pd.read_csv('../data/processed/v2_consumption_profiles.csv')

energy_dist = pd.read_csv('../data/processed/v2_energy _source_distribution.csv')

emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

usage_distribution_by_state = pd.read_csv(
    '../data/processed/v2_usage_distribution_by_state.csv',
    index_col=0
)

company_size_dist = pd.read_csv(
    '../data/processed/v2_company_size_distribution_by_usage.csv'
)

seasonality = pd.read_csv(
    '../data/processed/v2_seasonality_state_class_month.csv'
)

Sanity para integrar todos as tabelas estava dando MUITOS erros !

In [ ]:
profiles['usage_type'] = profiles['usage_type'].str.lower()

usage_distribution_by_state.columns = usage_distribution_by_state.columns.str.lower()

company_size_dist['Classe'] = company_size_dist['Classe'].str.lower()

seasonality['Classe'] = seasonality['Classe'].str.lower()

In [ ]:
print('profiles: '+profiles.columns,'\n')
print('energy_dist: '+energy_dist.columns,'\n')
print('emission_df: '+emission_df.columns,'\n')
print('usage_distribution_by_state: '+usage_distribution_by_state.columns,'\n')
print('company_size_dist: '+company_size_dist.columns,'\n')
print('seasonality: '+seasonality.columns,'\n')

In [ ]:
def show_uniques(df, name):
    print(f"\n{name}")
    for col in df.select_dtypes(include='object').columns:
        print(col, "->", df[col].unique())

In [ ]:
show_uniques(profiles, "profiles")
show_uniques(company_size_dist, "company_size_dist")
show_uniques(usage_distribution_by_state, "usage_distribution")
show_uniques(seasonality, "seasonality")

In [ ]:
print(set(profiles['usage_type']))
print(set(company_size_dist['Classe']))
print(set(usage_distribution_by_state.columns))
print(set(seasonality['Classe']))

In [ ]:
print("usage_types possíveis:", usage_distribution_by_state.columns.tolist())
print("classes em company_size:", company_size_dist['Classe'].unique())

In [ ]:
for u in usage_distribution_by_state.columns:
    subset = company_size_dist[company_size_dist['Classe'] == u]
    print(u, "->", len(subset))

In [ ]:
print(energy_dist['probability'].sum())

In [ ]:
print(company_size_dist.groupby('Classe')['probability'].sum())

função geração consumo

In [ ]:
def generate_consumption(profile_row):

    if profile_row['distribution_type'] == 'lognormal':

        mu = profile_row['param_1']
        sigma = profile_row['param_2']

        value = np.random.lognormal(mu, sigma)

        max_value = 10 * np.exp(mu)
        value = min(value, max_value)

        return value

    else:
        raise ValueError("Distribuição não suportada")

fonte energetica

In [ ]:
def sample_energy_source(dist_df):

    return dist_df.sample(
        1,
        weights=dist_df['probability']
    ).iloc[0]['energy_source']

calcular emissão

In [ ]:
def calculate_emission(energy_kwh, energy_source, emission_df):

    row = emission_df.loc[
        emission_df['energy_source'] == energy_source
    ]

    factor = row['emission_factor'].values[0]

    return energy_kwh * factor

amostra tamanho da empresa

In [ ]:
def sample_company_size(usage_type):

    subset = company_size_dist[company_size_dist['Classe'] == usage_type]

    if len(subset) == 1:
        return subset['company_size'].iloc[0]

    return np.random.choice(
        subset['company_size'],
        p=subset['probability']
    )

aplicar sazionalidade

In [ ]:
def apply_seasonality(consumption, state, usage_class, month):

    subset = seasonality[
        (seasonality['UF'] == state) &
        (seasonality['Classe'] == usage_class.capitalize()) &
        (seasonality['month'] == month)
    ]

    if subset.empty:
        return consumption

    factor = subset['seasonal_factor'].values[0]

    return consumption * factor

gerar evento (juntar tudo)

In [ ]:
def generate_event(profiles):

    # -------------------------------------------------
    # 1) Escolha do estado
    # -------------------------------------------------
    state = np.random.choice(states, p=state_weights)


    # -------------------------------------------------
    # 2) Sorteio do mês
    # -------------------------------------------------
    month = np.random.randint(1, 13)


    # -------------------------------------------------
    # 3) Setor econômico condicionado ao estado
    # -------------------------------------------------
    usage_probs_state = usage_distribution_by_state.loc[state].values
    usage_probs_state = usage_probs_state / usage_probs_state.sum()

    usage_types = usage_distribution_by_state.columns

    usage_type = np.random.choice(
        usage_types,
        p=usage_probs_state
    )


    # -------------------------------------------------
    # 4) Perfil do setor
    # -------------------------------------------------
    profiles_subset = profiles[profiles['usage_type'] == usage_type]

    if profiles_subset.empty:
        raise ValueError(f"Nenhum profile encontrado para {usage_type}")

    row = profiles_subset.sample(1).iloc[0]


    # -------------------------------------------------
    # 5) Consumo base
    # -------------------------------------------------
    consumption = generate_consumption(row)


    # -------------------------------------------------
    # 6) Ajuste de sazonalidade
    # -------------------------------------------------
    consumption = apply_seasonality(
        consumption,
        state,
        usage_type,
        month
    )


    # -------------------------------------------------
    # 7) RUÍDO REALISTA DE CONSUMO
    # -------------------------------------------------
    # Mesmo equipamentos idênticos nunca operam exatamente
    # no mesmo regime. Pequenas variações operacionais,
    # carga parcial e comportamento do usuário geram
    # flutuações naturais no consumo energético.
    #
    # Usamos um ruído multiplicativo moderado (~8%).
    #
    operational_variation = np.random.normal(1, 0.08)
    consumption = consumption * operational_variation


    # -------------------------------------------------
    # 8) Tamanho da empresa
    # -------------------------------------------------
    company_size = sample_company_size(usage_type)


    # -------------------------------------------------
    # 9) Fonte energética
    # -------------------------------------------------
    energy_source = sample_energy_source(energy_dist)


    # -------------------------------------------------
    # 10) Emissão base (sua lógica atual)
    # -------------------------------------------------
    co2 = calculate_emission(consumption, energy_source, emission_df)


    # -------------------------------------------------
    # 11) VARIAÇÃO REAL DO FATOR DE EMISSÃO
    # -------------------------------------------------
    # Mesmo dentro da mesma fonte energética existe
    # variação operacional:
    #
    # • qualidade do combustível
    # • eficiência da planta
    # • mistura de geração
    # • condições de operação
    #
    # Isso é modelado como uma pequena variação
    # no fator efetivo de emissão (~6%).
    #
    emission_factor_variation = np.random.normal(1, 0.06)
    co2 = co2 * emission_factor_variation


    # -------------------------------------------------
    # 12) RUÍDO FINAL DE MEDIÇÃO / PROCESSO
    # -------------------------------------------------
    # Última camada de ruído representa:
    #
    # • imprecisão de medição
    # • pequenas perdas do sistema
    # • arredondamentos operacionais
    #
    measurement_noise = np.random.normal(1, 0.03)
    co2 = co2 * measurement_noise


    # -------------------------------------------------
    # 13) Data dentro do mês sorteado
    # -------------------------------------------------
    date = (
        pd.Timestamp('2025-01-01')
        + pd.DateOffset(months=month-1)
        + pd.to_timedelta(np.random.randint(0,28), unit='D')
    )


    # -------------------------------------------------
    # 14) Retorno do evento
    # -------------------------------------------------
    return {

        'company_id': f"C{np.random.randint(100000,999999)}",

        'date': date,

        'state': state,

        'usage_type': usage_type,

        'company_size': company_size,

        'fuel_type': row['fuel_type'],

        'energy_kwh': consumption,

        'energy_source': energy_source,

        'co2_emission': co2
    }

gerar dataset

In [ ]:
def generate_dataset(n):

    data = []

    for _ in range(n):
        data.append(generate_event(profiles))

    return pd.DataFrame(data)

Montar dataset com tudo

In [ ]:
df = generate_dataset(100000)

df.head()

In [ ]:
df.to_csv('synthetic_energy_emissions_dataset.csv')